# Tutorial 7 — Feature extraction versus fine tuning

Use PyTorch as requested in the task. Compare frozen VGG16, frozen ResNet50, fine-tuned VGG16, fine-tuned ResNet50 and a custom CNN on identical CIFAR-10 splits.

**How to run:** select a Python kernel with the repository requirements, then Run All. QUICK mode keeps experiments short. For the larger run set `TUTORIAL_FULL=1` before starting the kernel. A quick run demonstrates the method; it is not a final accuracy benchmark.

**Execution status:** Complete implementation; end-to-end training/inference has not been verified in this repository run because the required large downloads did not finish. See `VALIDATION.md` for the separate offline model checks. Run All in Colab to generate your own results.

In [ ]:
# Google Colab: fetch the support files and install missing dependencies.
# Local Jupyter: follow README setup once; this cell makes no changes.
import sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules:
    repo = Path("/content/dl-course-tutorials")
    if not repo.exists():
        subprocess.check_call(["git", "clone", "https://github.com/naveed952/Tutorial-01_Perceptron_py.ipynb.git", str(repo)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo/"requirements.txt")])
    import os
    os.chdir(repo)


## Step 1 — Locate the project and set a reproducible seed
A fixed seed makes comparisons easier to reproduce. Each notebook can run independently.

In [ ]:
from pathlib import Path
import sys, os
ROOT = Path.cwd() if (Path.cwd()/"tutorial_support").exists() else Path.cwd().parent
assert (ROOT/"tutorial_support").exists(), "Run from the repository or notebooks folder."
sys.path.insert(0, str(ROOT))
from tutorial_support.common import *
seed()
print("Mode:", "QUICK: small real-data run" if QUICK else "FULL: larger training run")

## Step 2 — Load the dataset with the correct preprocessing
All runs use the same sample indices. Pretrained models need ImageNet normalization. QUICK uses 200 training images and 80 validation images, so results are demonstrations rather than reliable model rankings.

In [ ]:
import torch,copy,time,gc
from torch import nn
from torch.utils.data import DataLoader,Subset
from torchvision import datasets,models,transforms
import pandas as pd
DEVICE=torch_setup()
transform=transforms.Compose([transforms.Resize((64,64) if QUICK else (224,224)),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
data=datasets.CIFAR10(DATA,train=True,download=True,transform=transform)
testdata=datasets.CIFAR10(DATA,train=False,download=True,transform=transform)
ids=np.random.default_rng(SEED).permutation(len(data));ntr=200 if QUICK else 5000;nv=80 if QUICK else 1000
train=DataLoader(Subset(data,ids[:ntr].tolist()),batch_size=16,shuffle=True,num_workers=0)
val=DataLoader(Subset(data,ids[40000:40000+nv].tolist()),batch_size=16,num_workers=0)
test=DataLoader(Subset(testdata,list(range(80 if QUICK else 2000))),batch_size=16,num_workers=0)

## Step 3 — Define frozen backbones and new classifiers
VGG uses its convolutional feature extractor followed by global pooling and a small 10-class head, avoiding its large ImageNet Dense layers. ResNet uses its normal features and a replacement fc. Frozen means requires_grad=False, not that a forward pass is skipped.

In [ ]:
class TransferNet(nn.Module):
    def __init__(self,name):
        super().__init__();self.name=name
        if name=='VGG16':
            original=models.vgg16(weights=models.VGG16_Weights.DEFAULT)
            self.base=original.features;self.head=nn.Linear(512,10)
        else:
            original=models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            self.base=nn.Sequential(*list(original.children())[:-2]);self.head=nn.Linear(2048,10)
        self.pool=nn.AdaptiveAvgPool2d(1)
        for p in self.base.parameters():p.requires_grad=False
    def forward(self,x):return self.head(self.pool(self.base(x)).flatten(1))
    def unfreeze_last_block(self):
        block=self.base[-7:] if self.name=='VGG16' else self.base[-1]
        for p in block.parameters():p.requires_grad=True
    def train(self,mode=True):
        super().train(mode)
        # Keep ImageNet batch statistics fixed on this small dataset.
        for m in self.base.modules():
            if isinstance(m,nn.BatchNorm2d):m.eval()
        return self
class CustomCNN(nn.Module):
    def __init__(self):
        super().__init__();self.net=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(64,10))
    def forward(self,x):return self.net(x)

## Step 4 — Train with validation and early stopping
CrossEntropyLoss takes raw logits and integer class IDs: do not apply softmax first. Save the lowest-validation-loss checkpoint. Training and validation must use different modes.

In [ ]:
def epoch(model,loader,opt=None):
    model.train(opt is not None);total=correct=count=0
    with torch.set_grad_enabled(opt is not None):
        for x,y in loader:
            x,y=x.to(DEVICE),y.to(DEVICE);logits=model(x);loss=nn.functional.cross_entropy(logits,y)
            if opt is not None:opt.zero_grad();loss.backward();opt.step()
            total+=float(loss.detach())*len(y);correct+=int((logits.argmax(1)==y).sum());count+=len(y)
    return total/count,correct/count
def fit(model,lr):
    model.to(DEVICE);opt=torch.optim.Adam([p for p in model.parameters() if p.requires_grad],lr=lr,weight_decay=1e-4)
    best=float('inf');state=None;bad=0;hist={'loss':[],'val_loss':[],'accuracy':[],'val_accuracy':[]}
    for e in range(2 if QUICK else 10):
        tl,ta=epoch(model,train,opt);vl,va=epoch(model,val)
        for k,v in zip(hist,[tl,vl,ta,va]):hist[k].append(v)
        if vl<best:best=vl;state=copy.deepcopy(model.state_dict());bad=0
        else:bad+=1
        if bad>=3:break
    model.load_state_dict(state);return hist

## Step 5 — Run both freeze and fine-tune tasks plus a custom baseline
Fine-tuning starts from the trained frozen-head model, then unfreezes the upper block and lowers the learning rate. This controls the initialization better than comparing unrelated models.

In [ ]:
rows=[];histories={};best_loss=float('inf');best_state=None;best_kind=None
for kind in ['VGG16','ResNet50','Custom']:
    torch.manual_seed(SEED);model=CustomCNN() if kind=='Custom' else TransferNet(kind)
    for stage in (['scratch'] if kind=='Custom' else ['frozen','fine_tuned']):
        if stage=='fine_tuned':model.unfreeze_last_block()
        start=time.perf_counter();h=fit(model,1e-5 if stage=='fine_tuned' else 1e-3)
        name=kind+' '+stage;histories[name]=h;vl,va=epoch(model,val)
        rows.append({'model':name,'val_loss':vl,'val_accuracy':va,'trainable':sum(p.numel() for p in model.parameters() if p.requires_grad),'seconds':time.perf_counter()-start})
        if vl<best_loss:best_loss=vl;best_state={k:v.cpu().clone() for k,v in model.state_dict().items()};best_kind=kind;selected=name
    del model;gc.collect()
display(pd.DataFrame(rows));curves(histories);curves(histories,'accuracy')
winner=CustomCNN() if best_kind=='Custom' else TransferNet(best_kind);winner.load_state_dict(best_state);winner.to(DEVICE)
final=epoch(winner,test);print('Validation-selected:',selected,'test loss/accuracy:',final)
save_result('07_transfer',{'comparison':rows,'selected':selected,'test':final})

## Task discussion and improvement plan
Compare frozen ResNet with frozen VGG and the custom model; compare each backbone before and after fine-tuning. To improve a model, increase the data budget, preserve model-specific preprocessing, add training-only augmentation, tune the learning rate on validation, and unfreeze more layers only if needed. The supplied loop already includes weight decay and early stopping.

[PyTorch transfer-learning tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html).

## Explain it in class
Describe the input, target, model, loss, update rule and evaluation split. Use the displayed results from your own run. Changing the dataset, seed or training budget can change the result.

**Course source:** the matching tutorial PDF supplied in the Deep Learning course. Original explanatory implementation; the course PDFs are not redistributed.

## Task answers and revision check

**Feature extraction:** freeze the pretrained base and train a new classifier. **Fine-tuning:** start with the trained head, unfreeze selected upper layers and train with a smaller rate. Here the rate changes from 0.001 to 0.00001.

**How are comparisons controlled?** The notebook shares the same train/validation/test indices across VGG16, ResNet50 and the custom CNN. Fine-tuning continues from each backbone's trained frozen-head model. The quick image size and data budget are disclosed.

**Why freeze batch-normalization running statistics?** Small batches and a small dataset can overwrite useful pretrained statistics. requires_grad=False alone freezes weights, not all state updates.

**How could the result improve?** More representative data, training-only augmentation, a validation-selected learning rate, longer training with early stopping, and controlled unfreezing. Do not change all factors simultaneously when explaining their effects.

**Result to record after running:** frozen VGG/ResNet scores, fine-tuned VGG/ResNet scores, custom baseline, trainable parameters and training time. No pretrained classification accuracy is claimed in the current unexecuted notebook.
